In [4]:
import pandas as pd
import datetime

In [5]:
def read_tbl():
    # cols = ["first_name", "middle_name", ""]
    df = pd.read_csv("../data/input/25-467.csv")
    df = df.rename(columns={"uid": "person_nbr"})

    # df = df.rename(columns={"agency": "agency_name"})

    return df

def clean_sep_reason(df):
    df.loc[:, "separation_reason"] = (df
                                      .separation_reason
                                      .str.lower()
                                      .str.strip()
                                      .fillna("")
                                      .str.replace(r" l\. e\.", "", regex=True)
                                      .str.replace(r"dep\b", "department", regex=True)
                                      .str.replace(r"^charged(.+)", "criminal conviction", regex=True)
                                      .str.replace(r" \(explain\)", "", regex=True)

                                      )
    return df 


def fix_dates(df):
    df.loc[:, "start_date"] = pd.to_datetime(df.start_date)
    df.loc[:, "end_date"] = pd.to_datetime(df.end_date)

    df.loc[:, "start_date"] = (df.start_date
                               .astype(str)
                               .str.replace(r" ?00:00:00", "", regex=True))
    
    df.loc[:, "end_date"] = (df.end_date
                               .astype(str)
                               .str.replace(r" ?00:00:00", "", regex=True))
    return df 


def clean_date(date_str: str):
    try:
        year = int(date_str[:4])
        if year < 1800 or year > 2100:  # Adjust the range as needed
            return None
        return date_str
    except ValueError:
        return None
    

def clean_agency_name(df):
    df.loc[:, "agency_name"]  = (df.agency_name
                                 .str.lower()
                                 .str.strip()
                                 .str.replace(r" dist ", " district ", regex=True)
                                 .str.replace(r"dept\b", "department", regex=True)
                                 .str.replace(r"co\b", "county", regex=True)
                                 .str.replace(r"railrd", "railroad", regex=True)
                                 .str.replace(r"(\w+)  (\w+)", r"\1 \2", regex=True)
                                 .str.replace(r"il\b", "illinois", regex=True)
                                 .str.replace(r"&", "and", regex=True)
                                 .str.replace(r"c c", "community college", regex=True)
                                 .str.replace(r"univ\b", "university", regex=True)
                                 .str.replace(r"comm\. college", "community college", regex=True)
                                 .str.replace(r"u of i", "university of illinois", regex=True)
                                 .str.replace(r"supreme crt", "supreme court", regex=True)
    )
    return df[~((df.agency_name.str.contains("fire")))]

def collapse_contiguous_stints(df: pd.DataFrame) -> pd.DataFrame:
    by_cols = ["person_nbr", "first_name", "last_name", "agency_name"]
    df = df.fillna("")

    # assume missing end dates are current employment, and use today's date for
    # sorting purposes
    # assert df.start_date.notna().all()

    one_day = pd.to_timedelta(1, "days")
    today = pd.to_datetime(datetime.date.today(), utc=False)
    ancient = pd.to_datetime("1800-01-01", utc=False)
    working = df.sort_values(
        ["person_nbr", "agency_name", "start_date"], inplace=False
    )
    working["start_date"] = working["start_date"].apply(clean_date)
    working["end_date"] = working["end_date"].apply(clean_date)
    working["start_date"] = pd.to_datetime(
        working.start_date, utc=False
    ).fillna(ancient)

    working["end_date"] = pd.to_datetime(working.end_date
    , utc=False).fillna(
        today
    )
    working.loc[working.start_date < ancient, "start_date"] = ancient
    working.loc[working.end_date > today, "end_date"] = today
    grouped = working.groupby(by_cols)
    working["prv_end"] = grouped["end_date"].shift(1, fill_value=today)
    working["new_stint"] = (working.start_date - working.prv_end) > one_day
    working["stint_id"] = grouped["new_stint"].cumsum()
    collapsible = working.groupby(by_cols + ["stint_id"])
    summaries = {k: lambda x: x.tail(1) for k in df.columns if k not in by_cols}
    summaries["start_date"] = "min"
    summaries["end_date"] = "max"
    out = collapsible.aggregate(summaries).reset_index()
    out["start_date"] = out["start_date"].dt.strftime("%Y-%m-%d")
    out["end_date"] = out["end_date"].dt.strftime("%Y-%m-%d")
    out.loc[out.end_date == today.strftime("%Y-%m-%d"), "end_date"] = None
    out.loc[out.start_date == ancient.strftime("%Y-%m-%d"), "start_date"] = None
    return out.drop(["stint_id"], axis=1, inplace=False)



df = read_tbl()

df = df.pipe(clean_sep_reason).pipe(fix_dates).pipe(clean_agency_name)

df = df.sort_values("person_nbr", ascending=False)


In [6]:
df.agency_name.unique().tolist()

['dolton police department',
 'northwestern university police department',
 'north chicago police department',
 'elk grove village police department',
 "knox county sheriff's office",
 'university of chicago police department',
 'lincolnwood police department',
 'oak park police department',
 'burlington northern/santa fe railroad',
 'cook county forest pres district police',
 'broadview police department',
 "sangamon county sheriff's office",
 'matteson police department',
 'leland grove police department',
 'eldorado police department',
 'elgin police department',
 'palatine police department',
 'coal city police department',
 'evergreen park police department',
 "mchenry county sheriff's office",
 'oak lawn police department',
 "dupage county sheriff's office",
 'decatur police department',
 'park ridge police department',
 'kankakee police department',
 'bradley police department',
 'fox lake police department',
 'mount prospect police department',
 'forest park police department',

In [7]:
df

,person_nbr,last_name,first_name,middle_name,suffix,birth_year,race,sex,education,agency_name,agency_type,work_status,rank,start_date,end_date,separation_reason
183898,65167068,Stewart,Devonte,E,NaN,1993,Black or African American,Male,Some College,dolton police department,Law Enforcement,Full Time,Police Officer,2025-09-08,NaT,
183892,65167066,Kwon,Brian,Yongjoo,NaN,2000,Asian,Male,Bachelors Degree,northwestern university police department,Law Enforcement,Full Time,Police Officer,2025-09-01,NaT,
183891,65167065,Williams,Marcus,B,NaN,1992,Black or African American,Male,Some College,dolton police department,Law Enforcement,Full Time,Police Officer,2025-09-08,NaT,
183882,65167062,Guerrero,Yair,NaN,NaN,2004,Hispanic or Latino,Male,High School,north chicago police department,Law Enforcement,Full Time,Police Officer,2025-09-08,NaT,
183881,65167061,Coleman,Avontay,NaN,NaN,1999,Black or African American,Male,Masters Degree,north chicago police department,Law Enforcement,Full Time,Police Officer,2025-09-08,NaT,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138939,65000006,Aaron,Eugene,T.,NaN,1983,White,Male,Bachelors Degree,red bud police department,Law Enforcement,Full Time,Police Officer,2015-01-28,2016-05-17,resigned
135005,65000006,Aaron,Eugene,T.,NaN,1983,White,Male,Bachelors Degree,red bud police department,Law Enforcement,Part Time,Police Officer,2013-12-03,2015-01-19,other
91719,65000003,Aalto,Brian,John,NaN,1978,White,Male,NaN,mchenry police department,Law Enforcement,Full Time,Sergeant,2001-01-05,NaT,
37612,65000002,A'Hearn,Jo,Ann,Sr,1946,White,Female,High School,knox county sheriff's office,Law Enforcement,Full Time,Deputy,1979-04-09,2010-01-01,resigned
